# Complete MRI Processing Pipeline Visualization
## Longitudinal Infant Brain Development Analysis

This notebook demonstrates every step of the structural MRI processing pipeline on a single T1w image.

**Input File:** `/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz`

### Pipeline Overview:
1. **Load and Visualize Raw T1w Image**
2. **FreeSurfer Initial Processing** (autorecon1 up to step 15)
3. **Infant FreeSurfer Processing**
4. **Tissue Segmentation Merging** (iBEATv2 + iFS)
5. **White Matter Mask Generation**
6. **Surface Reconstruction** (autorecon2)
7. **Cortical Parcellation** (autorecon3)
8. **Final Measurements and Visualization**

In [ ]:
# Import required libraries
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.mplot3d import Axes3D
import subprocess
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Define paths and parameters
INPUT_T1W = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz"
SUBJECT_ID = "sub-01_ses-03"
AGE_MONTHS = 18  # Example age for infant FreeSurfer - adjust based on actual age

# Set up FreeSurfer environment
SUBJECTS_DIR = "/tmp/freesurfer_output"
IFS_DIR = "/tmp/iFS"
os.makedirs(SUBJECTS_DIR, exist_ok=True)
os.makedirs(IFS_DIR, exist_ok=True)

# Check if input file exists
if os.path.exists(INPUT_T1W):
    print(f"✓ Input file found: {INPUT_T1W}")
    file_size = os.path.getsize(INPUT_T1W) / (1024**2)  # MB
    print(f"  File size: {file_size:.2f} MB")
else:
    print(f"✗ Input file NOT found: {INPUT_T1W}")
    print("  Please update the INPUT_T1W path to your actual file location.")

## Step 1: Load and Visualize Raw T1w Image

Load the original T1-weighted anatomical image and visualize it in multiple planes.

In [ ]:
def load_nifti(filepath):
    """Load NIfTI file and return image data and header."""
    img = nib.load(filepath)
    data = img.get_fdata()
    return data, img.affine, img.header

def plot_3d_slices(data, title="MRI Slices", figsize=(15, 5), cmap='gray', percentiles=(1, 99)):
    """Plot axial, sagittal, and coronal slices of 3D MRI data."""
    # Calculate middle slices
    mid_x = data.shape[0] // 2
    mid_y = data.shape[1] // 2
    mid_z = data.shape[2] // 2
    
    # Calculate intensity range for better contrast
    vmin, vmax = np.percentile(data[data > 0], percentiles)
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Sagittal (side view)
    axes[0].imshow(np.rot90(data[mid_x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title(f'Sagittal (X={mid_x})')
    axes[0].axis('off')
    
    # Coronal (front view)
    axes[1].imshow(np.rot90(data[:, mid_y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title(f'Coronal (Y={mid_y})')
    axes[1].axis('off')
    
    # Axial (top view)
    axes[2].imshow(np.rot90(data[:, :, mid_z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title(f'Axial (Z={mid_z})')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Load original T1w image
if os.path.exists(INPUT_T1W):
    print("Loading original T1w image...")
    t1w_data, t1w_affine, t1w_header = load_nifti(INPUT_T1W)
    
    print(f"\nImage dimensions: {t1w_data.shape}")
    print(f"Voxel size: {t1w_header.get_zooms()[:3]} mm")
    print(f"Data type: {t1w_data.dtype}")
    print(f"Intensity range: [{t1w_data.min():.2f}, {t1w_data.max():.2f}]")
    
    # Plot the original image
    plot_3d_slices(t1w_data, title="Original T1w Image")
    plt.show()
else:
    print("Cannot proceed without input file. Please update the INPUT_T1W path.")

## Step 2: FreeSurfer Initial Processing (autorecon1)

This step performs:
- Motion correction
- Intensity normalization (nu correction)
- Talairach registration
- Intensity normalization to T1 reference
- Skull stripping

**Script reference:** `reFS_under25mo.sh:55` - `recon-all -all -subjid ${sub} -nonuintensitycor`

In [ ]:
def run_command(cmd, description="", show_output=False):
    """Run shell command and optionally show output."""
    print(f"\n{'='*60}")
    print(f"Running: {description}")
    print(f"{'='*60}")
    if show_output:
        result = subprocess.run(cmd, shell=True, capture_output=False, text=True)
    else:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
        else:
            print("✓ Completed successfully")
    return result.returncode

# Note: This would take several hours to run
# For demonstration, we'll show the command structure

print("""STEP 2: FreeSurfer Initial Processing

Command that would be run:
------------------------""")

fs_step1_cmd = f"""
export SUBJECTS_DIR={SUBJECTS_DIR}
recon-all -i {INPUT_T1W} \
          -subjid {SUBJECT_ID} \
          -all \
          -nonuintensitycor
"""

print(fs_step1_cmd)
print("\nThis step typically takes 6-12 hours.")
print("\nKey outputs generated:")
print("  - orig.mgz: Original input image in FreeSurfer format")
print("  - nu.mgz: Intensity normalized image")
print("  - T1.mgz: Further normalized T1 image")
print("  - brainmask.auto.mgz: Automated brain mask")
print("  - transforms/talairach.xfm: Talairach transformation")
print("  - aseg.auto.mgz: Automated tissue segmentation")
print("  - wm.mgz: White matter mask")

# For visualization purposes, we'll simulate having these files
# In actual use, you would run the command and wait for completion

### Visualize FreeSurfer Output (orig.mgz)

FreeSurfer converts the input to MGZ format and performs reorientation.

In [ ]:
# Check if FreeSurfer output exists
orig_mgz_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "mri", "orig.mgz")

if os.path.exists(orig_mgz_path):
    print("Loading FreeSurfer orig.mgz...")
    orig_data, _, _ = load_nifti(orig_mgz_path)
    plot_3d_slices(orig_data, title="FreeSurfer orig.mgz (Reoriented)")
    plt.show()
else:
    print(f"FreeSurfer output not found at: {orig_mgz_path}")
    print("This file would be created by running recon-all.")
    print("\nShowing expected file structure:")
    print(f"""
{SUBJECTS_DIR}/{SUBJECT_ID}/
├── mri/
│   ├── orig.mgz              # Original input
│   ├── nu.mgz                # Intensity normalized
│   ├── T1.mgz                # T1-normalized
│   ├── brainmask.mgz         # Brain mask
│   ├── aseg.mgz              # Tissue segmentation
│   └── transforms/
│       └── talairach.xfm     # Talairach transform
├── surf/                      # Surface files
├── label/                     # Label files
└── stats/                     # Statistics
    """)

## Step 3: Prepare for Infant FreeSurfer

**Script reference:** `reFS_under25mo.sh:59-67`

This step:
1. Removes certain FreeSurfer files that will be regenerated
2. Converts orig.mgz to mprage.nii.gz for infant FreeSurfer
3. Sets up the infant FreeSurfer directory structure

In [ ]:
print("""STEP 3: Prepare for Infant FreeSurfer

Commands that would be run:
""")

prep_ifs_cmds = f"""
# Remove certain files from initial FreeSurfer run
rm {SUBJECTS_DIR}/{SUBJECT_ID}/mri/transforms/*
rm {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig_nu.mgz
rm {SUBJECTS_DIR}/{SUBJECT_ID}/mri/mri_nu_correct.mni.log

# Create infant FreeSurfer directory
mkdir -p {IFS_DIR}/{SUBJECT_ID}

# Convert orig.mgz to mprage.nii.gz for infant FreeSurfer
mri_convert -i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig.mgz \
            -o {IFS_DIR}/{SUBJECT_ID}/mprage.nii.gz
"""

print(prep_ifs_cmds)

print("\nThis prepares the data for infant-specific processing.")

## Step 4: Infant FreeSurfer Processing

**Script reference:** `iFS_wrap.sh:14` - `infant_recon_all --s ${2} --age ${3}`

Infant FreeSurfer performs age-specific brain segmentation and processing optimized for developing brains.
It uses age-specific atlases and parameters to improve accuracy for infants.

In [ ]:
print(f"""STEP 4: Infant FreeSurfer Processing

This uses infant-specific atlases for age {AGE_MONTHS} months.

Command that would be run:
""")

ifs_cmd = f"""
# Set infant FreeSurfer environment
export FREESURFER_HOME=/path/to/infant_freesurfer
export SUBJECTS_DIR={IFS_DIR}

# Run infant FreeSurfer
infant_recon_all --s {SUBJECT_ID} --age {AGE_MONTHS}
"""

print(ifs_cmd)

print("\nThis step typically takes 4-8 hours.")
print("\nKey outputs:")
print("  - aseg.nii.gz: Age-appropriate tissue segmentation")
print("  - brainmask.mgz: Infant-specific brain mask")
print("  - transforms/talairach*.xfm: Infant atlas transformations")
print("\nInfant FreeSurfer provides better segmentation for:")
print("  - Thalamus")
print("  - Basal ganglia")
print("  - Other subcortical structures")
print("  - Developing cortical gray/white matter boundaries")

### Visualize Infant FreeSurfer Segmentation

The infant FreeSurfer aseg.nii.gz contains labeled tissue segmentation.

In [ ]:
def plot_segmentation(seg_data, title="Segmentation", figsize=(15, 5)):
    """Plot segmentation with color map."""
    # FreeSurfer color lookup table (simplified)
    from matplotlib.colors import ListedColormap
    
    # Create custom colormap for segmentation
    n_labels = int(seg_data.max()) + 1
    cmap = plt.cm.get_cmap('tab20', n_labels)
    
    mid_x = seg_data.shape[0] // 2
    mid_y = seg_data.shape[1] // 2
    mid_z = seg_data.shape[2] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    axes[0].imshow(np.rot90(seg_data[mid_x, :, :]), cmap=cmap, interpolation='nearest')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(np.rot90(seg_data[:, mid_y, :]), cmap=cmap, interpolation='nearest')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(np.rot90(seg_data[:, :, mid_z]), cmap=cmap, interpolation='nearest')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Check for infant FreeSurfer output
ifs_aseg_path = os.path.join(IFS_DIR, SUBJECT_ID, "mri", "aseg.mgz")

if os.path.exists(ifs_aseg_path):
    print("Loading infant FreeSurfer segmentation...")
    ifs_aseg_data, _, _ = load_nifti(ifs_aseg_path)
    
    print(f"Number of unique labels: {len(np.unique(ifs_aseg_data))}")
    print(f"Label range: {ifs_aseg_data.min():.0f} to {ifs_aseg_data.max():.0f}")
    
    plot_segmentation(ifs_aseg_data, title="Infant FreeSurfer Segmentation (aseg)")
    plt.show()
    
    # Show label statistics
    unique, counts = np.unique(ifs_aseg_data, return_counts=True)
    print("\nTop 10 labels by volume:")
    sorted_idx = np.argsort(counts)[::-1][:10]
    for idx in sorted_idx:
        label = unique[idx]
        count = counts[idx]
        print(f"  Label {label:3.0f}: {count:8d} voxels")
else:
    print(f"Infant FreeSurfer output not found at: {ifs_aseg_path}")
    print("This would be created by running infant_recon_all.")

## Step 5: Merge iBEATv2 and Infant FreeSurfer Segmentations

**Script reference:** `reFS_under25mo.sh:80` and `ibeat2aseg.m`

This critical step combines:
- **iBEATv2** tissue segmentation (superior cortical GM/WM/CSF labels)
- **Infant FreeSurfer** subcortical segmentation (superior subcortical structures)

The merging strategy:
1. Use iBEATv2 for cortical tissue types (1=CSF, 2=GM, 3=WM)
2. Use infant FreeSurfer for subcortical structures (thalamus, basal ganglia, etc.)
3. Resolve conflicts by nearest-neighbor labeling

In [ ]:
print("""STEP 5: Tissue Segmentation Merging

This step merges two segmentations:
  1. iBEATv2: Excellent cortical gray/white matter boundaries
  2. Infant FreeSurfer: Superior subcortical structure segmentation

Input required:
  - iBEATv2 tissue segmentation: /path/to/ibeat_tissue_seg.nii.gz
    Labels: 1=CSF, 2=GM, 3=WM

Algorithm (from ibeat2aseg.m):
""")

print("""
1. Ensure same dimensions between iBEAT and iFS segmentations
2. Zero out skull/outside brain using iBEAT mask
3. For each iBEAT voxel:
   
   WM voxels (label=3):
   - Map to left/right hemisphere WM (2/41)
   - Map to cerebellar WM (7/46)
   - Handle brainstem/vermis
   
   GM voxels (label=2):
   - Keep iFS labels for subcortical GM:
     * Thalamus (9→10, 48→49)
     * Caudate, Putamen, Pallidum
     * Hippocampus, Amygdala
   - Use cortical GM labels (3/42) for cortex
   - Use cerebellar GM (8/47) for cerebellum
   
   CSF voxels (label=1):
   - Lateral ventricles (4/43)
   - Other ventricles (14/15)
   - Non-ventricular CSF (0)

4. For unassigned voxels:
   - Find nearest neighbor with appropriate label
   - Use Euclidean distance in 3D space

5. Enforce iFS subcortical labels in final output
""") 

print("\nMATLAB command that would be run:")
print("""
matlab -nodesktop -nosplash -r "
  addpath('/path/to/scripts');
  ibeat2aseg('/path/to/ibeat_tissue.nii.gz', 
             '/path/to/iFS_dir/sub-01', 
             '/path/to/FS_dir/sub-01/mri');
  exit;"
""")

print("\nOutput: aseg.presurf.mgz - Combined segmentation for FreeSurfer")

### Visualize Merged Segmentation

The merged segmentation combines the best of both approaches.

In [ ]:
# FreeSurfer label names (key structures)
FS_LABEL_NAMES = {
    0: 'Unknown/CSF',
    2: 'Left-Cerebral-WM',
    3: 'Left-Cerebral-GM',
    4: 'Left-Lateral-Ventricle',
    7: 'Left-Cerebellum-WM',
    8: 'Left-Cerebellum-GM',
    10: 'Left-Thalamus',
    11: 'Left-Caudate',
    12: 'Left-Putamen',
    13: 'Left-Pallidum',
    17: 'Left-Hippocampus',
    18: 'Left-Amygdala',
    41: 'Right-Cerebral-WM',
    42: 'Right-Cerebral-GM',
    43: 'Right-Lateral-Ventricle',
    46: 'Right-Cerebellum-WM',
    47: 'Right-Cerebellum-GM',
    49: 'Right-Thalamus',
    50: 'Right-Caudate',
    51: 'Right-Putamen',
    52: 'Right-Pallidum',
    53: 'Right-Hippocampus',
    54: 'Right-Amygdala',
}

merged_aseg_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "mri", "aseg.presurf.mgz")

if os.path.exists(merged_aseg_path):
    print("Loading merged segmentation...")
    merged_aseg_data, _, _ = load_nifti(merged_aseg_path)
    
    plot_segmentation(merged_aseg_data, title="Merged Segmentation (iBEATv2 + iFS)")
    plt.show()
    
    # Volume statistics
    unique, counts = np.unique(merged_aseg_data, return_counts=True)
    print("\nSegmentation volumes:")
    for label, count in zip(unique, counts):
        if label in FS_LABEL_NAMES:
            print(f"{FS_LABEL_NAMES[label]:30s}: {count:8d} voxels")
else:
    print(f"Merged segmentation not found at: {merged_aseg_path}")
    print("This would be created by ibeat2aseg.m")

## Step 6: Generate White Matter Mask

**Script reference:** `reFS_under25mo.sh:80` and `aseg2wm.m`

Creates white matter mask (wm.mgz) from the merged segmentation for surface reconstruction.

In [ ]:
print("""STEP 6: White Matter Mask Generation

Extracts white matter voxels from merged segmentation.

White matter includes:
  - Cerebral WM (labels 2, 41)
  - Cerebellar WM (labels 7, 46)
  - Corpus callosum
  - Internal capsule

MATLAB command:
  aseg2wm('aseg.presurf.nii');

Output: wm.mgz
""")

wm_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "mri", "wm.mgz")

if os.path.exists(wm_path):
    print("\nLoading white matter mask...")
    wm_data, _, _ = load_nifti(wm_path)
    
    # Calculate WM volume
    wm_voxels = np.sum(wm_data > 0)
    print(f"White matter voxels: {wm_voxels:,}")
    
    plot_3d_slices(wm_data, title="White Matter Mask", cmap='hot')
    plt.show()
else:
    print(f"\nWhite matter mask not found at: {wm_path}")

## Step 7: Resume FreeSurfer Processing (autorecon2)

**Script reference:** `fs_autorecon2_end.sh`

This step performs surface reconstruction:
1. Bias field correction with infant atlas
2. EM registration to infant atlas
3. Intensity normalization with mask
4. WM surface tessellation
5. Topology correction
6. Surface smoothing and inflation
7. Spherical mapping
8. White matter surface placement

In [ ]:
print("""STEP 7: Surface Reconstruction (autorecon2)

This step creates cortical surface models.

Sub-steps:
""")

autorecon2_steps = [
    ("7.1", "Bias Field Correction", 
     "mri_nu_correct.mni --i orig.mgz --o nu.mgz --uchar transforms/talairach.xfm --n 2 --ants-n4"),
    
    ("7.2", "Intensity Normalization (to T1)", 
     "mri_normalize -g 1 -seed 1234 -mprage nu.mgz T1.mgz"),
    
    ("7.3", "Brain Masking with iFS", 
     "mri_mask T1.mgz [iFS]/brainmask.mgz brainmask.mgz"),
    
    ("7.4", "EM Registration to Infant Atlas", 
     "mri_em_register -uns 3 -mask brainmask.mgz nu.mgz $FREESURFER_HOME/average/RB_all_2020-01-02.gca transforms/talairach.lta"),
    
    ("7.5", "Intensity Normalization with Segmentation", 
     "mri_normalize -seed 1234 -mprage -aseg aseg.presurf.mgz -mask brainmask.mgz norm.mgz brain.mgz"),
    
    ("7.6", "Brain Masking for Surfaces", 
     "mri_mask -T 5 brain.mgz brainmask.mgz brain.finalsurfs.mgz"),
    
    ("7.7", "WM Volume Filling", 
     "mri_fill -a ../scripts/ponscc.cut.log -xform transforms/talairach.lta -segmentation aseg.presurf.mgz wm.mgz filled.mgz"),
    
    ("7.8", "Tessellation (LH)", 
     "mri_tessellate filled-pretess255.mgz 255 ../surf/lh.orig.nofix"),
    
    ("7.9", "Tessellation (RH)", 
     "mri_tessellate filled-pretess127.mgz 127 ../surf/rh.orig.nofix"),
    
    ("7.10", "Topology Correction (LH)", 
     "mris_topo_fixer -mgz -warnings sub-01 lh"),
    
    ("7.11", "Topology Correction (RH)", 
     "mris_topo_fixer -mgz -warnings sub-01 rh"),
    
    ("7.12", "Surface Remeshing (LH)", 
     "mris_remesh --remesh --iters 3 --input lh.orig.premesh --output lh.orig"),
    
    ("7.13", "Surface Remeshing (RH)", 
     "mris_remesh --remesh --iters 3 --input rh.orig.premesh --output rh.orig"),
    
    ("7.14", "White Matter Surface Placement (LH)", 
     "mris_make_surfaces -output .preaparc -soap -orig_white orig -aseg aseg.presurf -whiteonly -mgz -T1 brain.finalsurfs sub-01 lh"),
    
    ("7.15", "White Matter Surface Placement (RH)", 
     "mris_make_surfaces -output .preaparc -soap -orig_white orig -aseg aseg.presurf -whiteonly -mgz -T1 brain.finalsurfs sub-01 rh"),
    
    ("7.16", "Surface Smoothing and Inflation (LH)", 
     "mris_smooth -n 3 -nw -seed 1234 lh.white.preaparc lh.smoothwm\nmris_inflate lh.smoothwm lh.inflated"),
    
    ("7.17", "Surface Smoothing and Inflation (RH)", 
     "mris_smooth -n 3 -nw -seed 1234 rh.white.preaparc rh.smoothwm\nmris_inflate rh.smoothwm rh.inflated"),
    
    ("7.18", "Curvature Calculation", 
     "mris_curvature -w -seed 1234 lh.white.preaparc\nmris_curvature -w -seed 1234 rh.white.preaparc"),
]

for step_num, description, command in autorecon2_steps:
    print(f"\n{step_num}. {description}")
    print(f"    {command}")

print("\n" + "="*60)
print("This complete process takes 8-15 hours.")
print("="*60)

### Visualize Surface Reconstruction

FreeSurfer generates surface files representing the white matter and pial surfaces.

In [ ]:
def load_surface(filepath):
    """Load FreeSurfer surface file."""
    try:
        from nibabel.freesurfer import read_geometry
        vertices, faces = read_geometry(filepath)
        return vertices, faces
    except Exception as e:
        print(f"Error loading surface: {e}")
        return None, None

def plot_surface_3d(vertices, faces, title="Surface", color='lightblue', alpha=0.8):
    """Plot 3D surface mesh."""
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create mesh
    mesh = Poly3DCollection(vertices[faces], alpha=alpha, facecolor=color, edgecolor='none')
    ax.add_collection3d(mesh)
    
    # Set limits
    ax.set_xlim(vertices[:, 0].min(), vertices[:, 0].max())
    ax.set_ylim(vertices[:, 1].min(), vertices[:, 1].max())
    ax.set_zlim(vertices[:, 2].min(), vertices[:, 2].max())
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Set viewing angle
    ax.view_init(elev=20, azim=45)
    
    return fig, ax

# Load white matter surfaces
lh_white_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "lh.white.preaparc")
rh_white_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "rh.white.preaparc")

if os.path.exists(lh_white_path) and os.path.exists(rh_white_path):
    print("Loading white matter surfaces...")
    lh_verts, lh_faces = load_surface(lh_white_path)
    rh_verts, rh_faces = load_surface(rh_white_path)
    
    if lh_verts is not None and rh_verts is not None:
        print(f"LH surface: {len(lh_verts)} vertices, {len(lh_faces)} faces")
        print(f"RH surface: {len(rh_verts)} vertices, {len(rh_faces)} faces")
        
        # Plot left hemisphere
        plot_surface_3d(lh_verts, lh_faces, "Left Hemisphere White Matter Surface", color='steelblue')
        plt.show()
        
        # Plot right hemisphere  
        plot_surface_3d(rh_verts, rh_faces, "Right Hemisphere White Matter Surface", color='coral')
        plt.show()
    else:
        print("Error loading surface geometry.")
else:
    print(f"Surface files not found.")
    print("These would be created during autorecon2.")
    print("\nExpected files:")
    print(f"  - {lh_white_path}")
    print(f"  - {rh_white_path}")

## Step 8: Cortical Parcellation (autorecon3)

**Script reference:** `fs_autorecon3_wrap.sh`

Final processing steps:
1. Spherical registration to atlas
2. Cortical parcellation (Desikan-Killiany atlas)
3. Pial surface placement with infant-specific parameters
4. Cortical ribbon generation
5. Statistical measurements

In [ ]:
print("""STEP 8: Cortical Parcellation and Final Processing (autorecon3)

Key steps:
""")

autorecon3_steps = [
    ("8.1", "Spherical Registration to Atlas",
     "Uses infant-specific atlas for age-appropriate parcellation"),
    
    ("8.2", "Cortical Parcellation",
     "Labels cortical regions using Desikan-Killiany atlas (34 regions per hemisphere)"),
    
    ("8.3", "Pial Surface Placement (with infant parameters)",
     "mris_make_surfaces -grad_dir 1 -intensity .3 -output .tmp -pial_offset .25 -nowhite -noaparc -aseg aseg.presurf -orig_pial white sub-01 lh"),
    
    ("8.4", "Cortical Ribbon Generation",
     "Creates volume between white and pial surfaces"),
    
    ("8.5", "Statistical Measurements",
     "Computes thickness, surface area, volume for each region"),
]

for step_num, description, details in autorecon3_steps:
    print(f"\n{step_num}. {description}")
    print(f"    {details}")

print("\n" + "="*60)
print("Note: Infant-specific parameters are used:")
print("  - grad_dir 1: Gradient direction")
print("  - intensity .3: Lower intensity threshold for infant brains")
print("  - pial_offset .25: Smaller offset for thinner infant cortex")
print("\nThis process takes 3-6 hours.")
print("="*60)

### Visualize Cortical Parcellation

In [ ]:
# Desikan-Killiany atlas labels
DK_LABELS = {
    1000: 'unknown', 1001: 'bankssts', 1002: 'caudalanteriorcingulate',
    1003: 'caudalmiddlefrontal', 1005: 'cuneus', 1006: 'entorhinal',
    1007: 'fusiform', 1008: 'inferiorparietal', 1009: 'inferiortemporal',
    1010: 'isthmuscingulate', 1011: 'lateraloccipital', 1012: 'lateralorbitofrontal',
    1013: 'lingual', 1014: 'medialorbitofrontal', 1015: 'middletemporal',
    1016: 'parahippocampal', 1017: 'paracentral', 1018: 'parsopercularis',
    1019: 'parsorbitalis', 1020: 'parstriangularis', 1021: 'pericalcarine',
    1022: 'postcentral', 1023: 'posteriorcingulate', 1024: 'precentral',
    1025: 'precuneus', 1026: 'rostralanteriorcingulate', 1027: 'rostralmiddlefrontal',
    1028: 'superiorfrontal', 1029: 'superiorparietal', 1030: 'superiortemporal',
    1031: 'supramarginal', 1032: 'frontalpole', 1033: 'temporalpole',
    1034: 'transversetemporal', 1035: 'insula'
}

# Load parcellation
lh_aparc_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "label", "lh.aparc.annot")
rh_aparc_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "label", "rh.aparc.annot")

if os.path.exists(lh_aparc_path) and os.path.exists(rh_aparc_path):
    print("Loading cortical parcellation...")
    
    from nibabel.freesurfer import read_annot
    
    lh_labels, lh_ctab, lh_names = read_annot(lh_aparc_path)
    rh_labels, rh_ctab, rh_names = read_annot(rh_aparc_path)
    
    print(f"\nLeft hemisphere: {len(np.unique(lh_labels))} regions")
    print(f"Right hemisphere: {len(np.unique(rh_labels))} regions")
    
    print("\nCortical regions identified:")
    for name in lh_names[:10]:  # Show first 10
        print(f"  - {name.decode('utf-8')}")
    print("  ...")
else:
    print("Parcellation files not found.")
    print("These would be created during autorecon3.")

### Visualize Pial Surface

In [ ]:
# Load pial surfaces
lh_pial_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "lh.pial")
rh_pial_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "rh.pial")

if os.path.exists(lh_pial_path) and os.path.exists(rh_pial_path):
    print("Loading pial surfaces...")
    lh_pial_verts, lh_pial_faces = load_surface(lh_pial_path)
    rh_pial_verts, rh_pial_faces = load_surface(rh_pial_path)
    
    if lh_pial_verts is not None:
        # Plot overlaid white and pial surfaces
        if os.path.exists(lh_white_path):
            fig = plt.figure(figsize=(15, 12))
            
            # Left hemisphere
            ax1 = fig.add_subplot(121, projection='3d')
            from mpl_toolkits.mplot3d.art3d import Poly3DCollection
            
            # White surface (inner)
            mesh_white = Poly3DCollection(lh_verts[lh_faces], alpha=0.3, 
                                        facecolor='steelblue', edgecolor='none')
            ax1.add_collection3d(mesh_white)
            
            # Pial surface (outer)
            mesh_pial = Poly3DCollection(lh_pial_verts[lh_pial_faces], alpha=0.5, 
                                       facecolor='coral', edgecolor='none')
            ax1.add_collection3d(mesh_pial)
            
            all_verts = np.vstack([lh_verts, lh_pial_verts])
            ax1.set_xlim(all_verts[:, 0].min(), all_verts[:, 0].max())
            ax1.set_ylim(all_verts[:, 1].min(), all_verts[:, 1].max())
            ax1.set_zlim(all_verts[:, 2].min(), all_verts[:, 2].max())
            ax1.set_title('Left Hemisphere\n(Blue: White, Red: Pial)', fontweight='bold')
            ax1.view_init(elev=20, azim=45)
            
            # Right hemisphere
            ax2 = fig.add_subplot(122, projection='3d')
            
            mesh_white_r = Poly3DCollection(rh_verts[rh_faces], alpha=0.3, 
                                          facecolor='steelblue', edgecolor='none')
            ax2.add_collection3d(mesh_white_r)
            
            mesh_pial_r = Poly3DCollection(rh_pial_verts[rh_pial_faces], alpha=0.5, 
                                         facecolor='coral', edgecolor='none')
            ax2.add_collection3d(mesh_pial_r)
            
            all_verts_r = np.vstack([rh_verts, rh_pial_verts])
            ax2.set_xlim(all_verts_r[:, 0].min(), all_verts_r[:, 0].max())
            ax2.set_ylim(all_verts_r[:, 1].min(), all_verts_r[:, 1].max())
            ax2.set_zlim(all_verts_r[:, 2].min(), all_verts_r[:, 2].max())
            ax2.set_title('Right Hemisphere\n(Blue: White, Red: Pial)', fontweight='bold')
            ax2.view_init(elev=20, azim=-45)
            
            plt.suptitle('White Matter and Pial Surfaces', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.show()
            
            # Calculate cortical thickness statistics
            thickness_estimate = np.mean(np.linalg.norm(lh_pial_verts - lh_verts, axis=1))
            print(f"\nEstimated average cortical thickness (LH): {thickness_estimate:.2f} mm")
else:
    print("Pial surface files not found.")
    print("These would be created during autorecon3.")

## Step 9: Extract Final Measurements

FreeSurfer computes comprehensive morphometric measurements stored in the stats/ directory.

In [ ]:
def read_stats_file(filepath):
    """Read FreeSurfer stats file."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            if not line.startswith('#') and line.strip():
                data.append(line.strip().split())
    return data

# Check for stats files
lh_aparc_stats = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "lh.aparc.stats")
rh_aparc_stats = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "rh.aparc.stats")
aseg_stats = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "aseg.stats")

print("STEP 9: Final Measurements\n")
print("FreeSurfer generates statistics for:")
print("\nCortical measurements (aparc.stats):")
print("  - Surface area (mm²) per region")
print("  - Gray matter volume (mm³) per region")
print("  - Cortical thickness (mm) per region")
print("  - Mean curvature per region")
print("  - Folding index per region")

print("\nSubcortical measurements (aseg.stats):")
print("  - Volume (mm³) for subcortical structures:")
print("    * Thalamus, Caudate, Putamen, Pallidum")
print("    * Hippocampus, Amygdala")
print("    * Ventricles")
print("    * Cerebellum")

print("\nGlobal measurements:")
print("  - Total brain volume")
print("  - Total gray matter volume")
print("  - Total white matter volume")
print("  - Total cortical surface area")
print("  - Average cortical thickness")

if os.path.exists(aseg_stats):
    print("\n" + "="*60)
    print("Loading actual measurements...")
    # Parse stats file
    with open(aseg_stats, 'r') as f:
        for line in f:
            if 'Brain Segmentation Volume' in line:
                print(line.strip())
            elif 'lhCortex' in line or 'rhCortex' in line:
                print(line.strip())
else:
    print("\nStats files would be generated in:")
    print(f"  {SUBJECTS_DIR}/{SUBJECT_ID}/stats/")

## Step 10: Visualize Complete Processing Summary

Create a comprehensive figure showing the complete pipeline.

In [ ]:
print("""COMPLETE PIPELINE SUMMARY
========================

INPUT:
  T1w.nii.gz - Raw anatomical image
  iBEATv2 tissue segmentation (prerequisite)
  Subject age (for infant FreeSurfer)

PROCESSING STEPS:

1. FreeSurfer Initial Processing (~6-12 hours)
   ├─ Motion correction and averaging
   ├─ Intensity normalization
   ├─ Talairach registration
   ├─ Skull stripping
   └─ Initial tissue segmentation

2. Infant FreeSurfer Processing (~4-8 hours)
   ├─ Age-specific atlas registration
   ├─ Infant-optimized segmentation
   └─ Superior subcortical labeling

3. Segmentation Merging (MATLAB, ~5-10 minutes)
   ├─ Combine iBEATv2 cortical labels
   ├─ Combine iFS subcortical labels
   ├─ Resolve conflicts with nearest-neighbor
   └─ Generate aseg.presurf.mgz

4. White Matter Mask Generation (MATLAB, ~1-2 minutes)
   └─ Extract WM voxels → wm.mgz

5. Surface Reconstruction (~8-15 hours)
   ├─ WM volume filling
   ├─ Tessellation
   ├─ Topology correction
   ├─ Surface smoothing
   ├─ Surface inflation
   └─ White matter surface placement

6. Cortical Parcellation (~3-6 hours)
   ├─ Spherical registration to atlas
   ├─ Cortical region labeling (34 regions/hemisphere)
   ├─ Pial surface placement (infant parameters)
   └─ Statistical measurements

OUTPUTS:

Volumes:
  ├─ orig.mgz - Original image
  ├─ brain.mgz - Skull-stripped brain
  ├─ aseg.mgz - Tissue segmentation (with subcortical labels)
  ├─ ribbon.mgz - Cortical ribbon
  └─ aparc+aseg.mgz - Combined parcellation

Surfaces:
  ├─ lh/rh.white - White matter surface
  ├─ lh/rh.pial - Pial surface
  ├─ lh/rh.inflated - Inflated surface
  └─ lh/rh.sphere - Spherical surface

Labels:
  └─ lh/rh.aparc.annot - Desikan-Killiany parcellation

Statistics:
  ├─ aseg.stats - Subcortical volumes
  ├─ lh/rh.aparc.stats - Cortical measurements per region
  │   ├─ Surface area (mm²)
  │   ├─ Gray matter volume (mm³)
  │   ├─ Cortical thickness (mm)
  │   ├─ Mean curvature
  │   └─ Folding index
  └─ Global measures

TOTAL PROCESSING TIME: ~20-40 hours (depending on hardware)

KEY FEATURES OF THIS PIPELINE:
✓ Optimized for infant brains (3-50 months)
✓ Combines strength of multiple tools (iBEATv2 + iFS + FS)
✓ Age-specific atlases and parameters
✓ Comprehensive morphometric measurements
✓ Longitudinal tracking capability
""")

## Running the Complete Pipeline

To run this pipeline on your data, you would:

1. **Prepare prerequisites:**
   ```bash
   # Run iBEATv2 to get tissue segmentation
   # (not included in this repository)
   ```

2. **Run the structural pipeline:**
   ```bash
   cd peer-review/1.Structure
   
   # For infants under 25 months:
   ./reFS_under25mo.sh sub-01_ses-03 \
       /path/to/ibeat_tissue_seg.nii.gz \
       18  # age in months
   
   # OR for children 25-50 months:
   ./reFS_25-50mo.sh sub-01_ses-03 \
       /path/to/ibeat_tissue_seg.nii.gz \
       30  # age in months
   ```

3. **Extract statistics:**
   ```bash
   ./consol_stats.sh
   ```

4. **Analyze results:**
   - View surfaces in FreeView: `freeview -f surf/lh.pial:annot=aparc`
   - Check statistics in `stats/` directory
   - Use R scripts in `3.Statistics/` for longitudinal analysis

## References and Resources

### Tools Used:
- **FreeSurfer 7.3+**: https://surfer.nmr.mgh.harvard.edu/
- **Infant FreeSurfer**: Specialized for infant brain processing
- **iBEATv2**: Infant Brain Extraction and Analysis Toolbox

### Key Papers:
1. Infant FreeSurfer processing methods
2. iBEATv2 infant brain segmentation
3. Longitudinal infant brain development trajectories

### FreeSurfer Documentation:
- Recon-all pipeline: https://surfer.nmr.mgh.harvard.edu/fswiki/recon-all
- Surface reconstruction: https://surfer.nmr.mgh.harvard.edu/fswiki/SurfaceReconstruction
- Cortical parcellation: https://surfer.nmr.mgh.harvard.edu/fswiki/CorticalParcellation